In [ ]:
#Installations
!pip install PyPDF2
!pip install PyMuPDF
!pip install keybert
!pip install rouge
!pip install textacy
!pip install pdfplumber
!pip install rdflib
!pip install bert-score
!pip install autocorrect
!pip install sentence-transformers
!pip install faiss-cpu --no-cache

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 53.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 59.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 18.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 29.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 34.3 MB/s eta 0:00:00
  Created wheel for keybert: filename=keybert-0.8.3-py3-none-any.whl size=39124 sha256=fc8b1c429da898fe3d73858de07

In [ ]:
!python -m spacy download en_core_web_lg

2023-10-25 18:55:23.198347: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-25 18:55:23.198399: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-25 18:55:23.198433: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-25 18:55:23.206322: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-25 18:55:24.290843: W tensorflow/compiler/

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import math
import IPython
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import numpy as np
import itertools
import unicodedata
import re
from re import sub
import matplotlib.pyplot as plt
import matplotlib.image as img
import string
from string import punctuation
import time
from pprint import pprint
from sentence_transformers import SentenceTransformer
import faiss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from autocorrect import Speller
spell = Speller()
from sentence_transformers import CrossEncoder

from rouge import Rouge

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.probability import FreqDist
from nltk.parse.corenlp import CoreNLPParser, CoreNLPDependencyParser
from nltk.tree import ParentedTree
from nltk.stem.snowball import SnowballStemmer

# Use English stemmer.
stemmer = SnowballStemmer("english")
from collections import Counter
from tqdm import tqdm

import copy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import spacy
nlp = spacy.load("en_core_web_lg")
from spacy.matcher import Matcher
from spacy.matcher import DependencyMatcher
from spacy.tokens import Span
from spacy import displacy

from rdflib.namespace import DC, DCTERMS, DOAP, FOAF, OWL, RDF, RDFS, SKOS, VOID, XMLNS, XSD
from rdflib import URIRef, BNode, Literal, Namespace, Graph
from rdflib.extras import describer

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import PyPDF2
import fitz
import re


# Replace 'your_pdf_file.pdf' with the actual path to your PDF file
pdf_file_path = '/content/drive/MyDrive/PMBOK6-2017_wf.pdf'

# Create a PDF object
pdf_file = open(pdf_file_path, 'rb')
pdf_reader = fitz.open(pdf_file)

In [ ]:
import fitz  # PyMuPDF

output_txt_file_path = 'PDF_LINES_NOT_CLEAN.txt'

start_page = 37
end_page = 524

output_txt_file = open(output_txt_file_path, 'w', encoding='utf-8')

# Open the PDF file
pdf_document = fitz.open(pdf_file_path)  # Replace 'your_pdf_file.pdf' with your PDF file path

for page_num in range(start_page - 1, end_page):
    page = pdf_document.load_page(page_num)

    # Extract text from the page
    page_text = page.get_text()

    # Split the page text into lines
    lines = page_text.split('\n')

    # Process each line
    for line in lines:
        print(line)
        output_txt_file.write(line + '\n')

# Close the PDF file
pdf_document.close()

# Close the output text file
output_txt_file.close()


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
the project. This information may be at the activity or project level. This includes consideration of attributes such as 
resource experience and/or skill level, as well as various geographical locations. 
 
9.3.3.4 CHANGE REQUESTS 
Described in Section 4.3.3.4. When changes occur as a result of carrying out the Acquire Resources process (for example, 
impacts to the schedule) or when recommended corrective or preventive actions impact any of the components of the 
project management plan or project documents, the project manager needs to submit a change request. Change requests 
are processed for review and disposition through the Perform Integrated Change Control process (Section 4.6). 
 
9.3.3.5 PROJECT MANAGEMENT PLAN UPDATES 
Any change to the project management plan goes through the organization’s change control process via a change 
request. Components of the project management plan that may be updated

In [ ]:
# #Read CSV
# import pandas as pd
# df = pd.read_csv('PDF_LINES_NOT_CLEAN.csv')

In [ ]:
file_path = 'PDF_LINES_NOT_CLEAN.txt'

try:
    with open(file_path, 'r',encoding='utf-8') as file:
        # Read the contents of the file
        file_contents = file.read()
        print(file_contents)
except FileNotFoundError:
    print(f"The file '{file_path}' was not found.")


1 
 
1 
 
INTRODUCTION  
 
1.1 OVERVIEW AND PURPOSE OF THIS GUIDE 
Project management is not new. It has been in use for hundreds of years. Examples of project outcomes include: 
◆ Pyramids of Giza, 
◆ Olympic games, 
◆ Great Wall of China, 
◆ Taj Mahal, 
◆ Publication of a children’s book, 
◆ Panama Canal, 
◆ Development of commercial jet airplanes, 
◆ Polio vaccine, 
◆ Human beings landing on the moon, 
◆ Commercial software applications, 
◆ Portable devices to use the global positioning system (GPS), and 
◆ Placement of the International Space Station into Earth’s orbit. 
The outcomes of these projects were the result of leaders and managers applying project management practices, 
principles, processes, tools, and techniques to their work. The managers of these projects used a set of key skills and 
applied knowledge to satisfy their customers and other people involved in and affected by the project. By the mid-20th 
century, project managers began the work of seeking recognition fo

In [ ]:
# Replace lines with X page
def replace_lines(input_text):
    i=1
    output_lines = []

    for line in input_text.split('\n'):
        match = re.search(str(i), line)
        if match:
            num = int(match.group())
            if line and num==i and line[0].isdigit():
                rest_of_line = line[len(str(num)):]
                i=i+1
                if num % 2 == 0:
                    # Even number, replace with "NubPage X"
                    output_lines.append(f"NubPage {num}\n{rest_of_line}")
                else:
                    # Odd number, replace with "NubPage X"
                    output_lines.append(f"NubPage {num}\n{rest_of_line}")

        else:
            output_lines.append(line)

    return '\n'.join(output_lines)


In [ ]:
lines = replace_lines(file_contents).split('\n')
for line in lines:
    print(line)

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
be used are: 
◆ Shared portal. A shared repository for information sharing (e.g., website, collaboration software or intranet) is 
effective for virtual project teams. 
◆ Video conferencing. Video conferencing is an important technique for effective communication with 
virtual teams. 
◆ Audio conferencing. Communication within a team using audio conferencing is another technique to build 
rapport and confidence within virtual teams. 
◆ Email/chat. Regular communications using email and chat is also an effective technique. 

NubPage 315
 
 
9.4.2.4 INTERPERSONAL AND TEAM SKILLS 
Interpersonal and team skills that can be used for this process include but are not limited to: 
◆ Conflict management. Described in Section 9.5.2.1. The project manager needs to resolve conflicts in a timely 
manner and in a constructive way in order to achieve a high-performing team. 
◆ Influencing. Described in Section 9.5.2.1. An i

In [ ]:
#REMOVE UNWANTED CHARCTERS
modified_lines = []
for line in lines:
    modified_line = line.replace("uu", "")
    modified_lines.append(modified_line)
output_text = '\n'.join(modified_lines)

In [ ]:
# Part 1 - Guide
modified_lines = []
lines=output_text.split('\n')
for line in lines:
    modified_line = line.replace("Part 1 - Guide", "")
    modified_lines.append(modified_line)
output_text = '\n'.join(modified_lines)

In [ ]:
# u n
modified_lines = []
lines=output_text.split('\n')
for line in lines:
    modified_line = line.replace("u n", "")
    modified_lines.append(modified_line)
output_text = '\n'.join(modified_lines)

In [ ]:
# "Not For Distribution, Sale or Reproduction."
modified_lines = []
lines=output_text.split('\n')
for line in lines:
    modified_line = line.replace('Not For Distribution, Sale or Reproduction.', "")
    modified_lines.append(modified_line)
output_text = '\n'.join(modified_lines)

In [ ]:
# delete empty lines

lines = output_text.split('\n')
non_empty_lines = [line for line in lines if line.strip() != ""]
output_text = '\n'.join(non_empty_lines)

In [ ]:
# Remove spaces from the beginning of the every line
lines = output_text.split('\n')
stripped_lines = [line.lstrip() for line in lines]
output_text = '\n'.join(stripped_lines)

In [ ]:
#need to clean it from "u m"

In [ ]:
lines = output_text.split('\n')
for line in lines:
        print(line, "$")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
in the project. Quality assurance is about using project processes effectively. It involves following and meeting standards  $
to assure stakeholders that the final product will meet their needs, expectations, and requirements. Manage Quality  $
includes all the quality assurance activities, and is also concerned with the product design aspects and process  $
improvements. Manage Quality work will fall under the conformance work category in the cost of quality framework.  $
NubPage 268 $
The Manage Quality process implements a set of planned and systematic acts and processes defined within the  $
project’s quality management plan that helps to:  $
◆ Design an optimal and mature product by implementing specific design guidelines that address specific aspects  $
of the product,  $
◆ Build confidence that a future output will be completed in a manner that meets the specified requirements and  $
expectations thro

In [ ]:
# #SAVE TO FILE .TXT
# output_txt_file_path = 'PDF_LINES_SEMI_CLEAN.txt'
# output_txt_file = open(output_txt_file_path, 'w',encoding='utf-8')
# for line in lines:
#     output_txt_file.write(line + '\n')

In [ ]:
#Split chapters (we have 13 chapters)

import  re
import pandas as pd
Chapters=[]
#Pages=[]
Contents=[]


#initial first chapter
Current_chap="0"
Page="1"
content=""
change_Page=False

#initial chapter 1's
Chapters.append(Current_chap)
#Pages.append(Page)

i=int(Current_chap)+1
Next_chap=str(i)

for line in lines:
    line=line.strip()
    #match = re.search(r"NubPage (\d+)", line)
    #Cheking Chapter
    if line and line==Next_chap :
        Contents.append(content)
        content=""
        change_Page=True
        Current_chap=Next_chap
        i=int(Current_chap)+1
        Next_chap=str(i)
        Chapters.append(Current_chap)
    #elif line and match  :
     #   if match and len(Pages)<i and change_Page:
      #      change_Page=False
       #     page = match.group(1)
        #    page_str = str(int(page) - 1)
            #print(f"Chapter '{Current_chap}'PAGE '{page_str}'")
         #   Pages.append(page_str)
    else:
        content += '\n' + line
print(Current_chap)
Contents.append(content)

# Make sure all lists have the same length by filling any missing data with placeholders
max_length = max(len(Chapters), len(Contents))
Chapters.extend([""] * (max_length - len(Chapters)))
#Pages.extend([""] * (max_length - len(Pages)))
Contents.extend([""] * (max_length - len(Contents)))

# Create the dataframe
df = pd.DataFrame({
    'Chapter': Chapters,
   # 'Page': Pages,
    'Content': Contents,
})

df

13


,Chapter,Content
0,0,\nNubPage 1
1,1,\nINTRODUCTION\n1.1 OVERVIEW AND PURPOSE OF TH...
2,2,\nTHE ENVIRONMENT IN WHICH PROJECTS OPERATE\n2...
3,3,\nTHE ROLE OF THE PROJECT MANAGER\n3.1 OVERVIE...
4,4,\nPROJECT INTEGRATION MANAGEMENT\nProject Inte...
5,5,\nPROJECT SCOPE MANAGEMENT\nProject Scope Mana...
6,6,\nPROJECT SCHEDULE MANAGEMENT\nProject Schedul...
7,7,\nPROJECT COST MANAGEMENT\nProject Cost Manage...
8,8,\nPROJECT QUALITY MANAGEMENT\nProject Quality ...
9,9,\nPROJECT RESOURCE MANAGEMENT\nProject Resourc...


In [ ]:
lines_chap1=lines

In [ ]:
content_chap1=''
for line in lines:
    content_chap1 += '\n' + line
df.at[0, "Content"]=content_chap1

#dropping the first row
df=df.drop(0)
#removing \n
df["Content"] = df["Content"].str.replace(r'^\n', '', 1)

df

,Chapter,Content
1,1,INTRODUCTION\n1.1 OVERVIEW AND PURPOSE OF THIS...
2,2,THE ENVIRONMENT IN WHICH PROJECTS OPERATE\n2.1...
3,3,THE ROLE OF THE PROJECT MANAGER\n3.1 OVERVIEW\...
4,4,PROJECT INTEGRATION MANAGEMENT\nProject Integr...
5,5,PROJECT SCOPE MANAGEMENT\nProject Scope Manage...
6,6,PROJECT SCHEDULE MANAGEMENT\nProject Schedule ...
7,7,PROJECT COST MANAGEMENT\nProject Cost Manageme...
8,8,PROJECT QUALITY MANAGEMENT\nProject Quality Ma...
9,9,PROJECT RESOURCE MANAGEMENT\nProject Resource ...
10,10,PROJECT COMMUNICATIONS MANAGEMENT\nProject Com...


In [ ]:
import pandas as pd
file_path = 'DF_Chapters_Pages_Content.csv'
# Export the DataFrame to a CSV file
df.to_csv(file_path, index=False)

In [ ]:
df_chapters = pd.read_csv(file_path)
df_chapters

,Chapter,Content
0,1,INTRODUCTION\n1.1 OVERVIEW AND PURPOSE OF THIS...
1,2,THE ENVIRONMENT IN WHICH PROJECTS OPERATE\n2.1...
2,3,THE ROLE OF THE PROJECT MANAGER\n3.1 OVERVIEW\...
3,4,PROJECT INTEGRATION MANAGEMENT\nProject Integr...
4,5,PROJECT SCOPE MANAGEMENT\nProject Scope Manage...
5,6,PROJECT SCHEDULE MANAGEMENT\nProject Schedule ...
6,7,PROJECT COST MANAGEMENT\nProject Cost Manageme...
7,8,PROJECT QUALITY MANAGEMENT\nProject Quality Ma...
8,9,PROJECT RESOURCE MANAGEMENT\nProject Resource ...
9,10,PROJECT COMMUNICATIONS MANAGEMENT\nProject Com...


In [ ]:
# Function to insert the first line from Content into Chapter and remove it from Content
def insert_first_line_to_chapter(row):
    first_line, rest_content = row['Content'].split('\n', 1)
    return str(row['Chapter']) + ' ' + first_line, rest_content

# Apply the function to update Chapter and Content columns
df_chapters[['Chapter', 'Content']] = df_chapters.apply(insert_first_line_to_chapter, axis=1).values.tolist()

df_chapters

,Chapter,Content
0,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE\nProjec...
1,2 THE ENVIRONMENT IN WHICH PROJECTS OPERATE,2.1 OVERVIEW\nProjects exist and operate in en...
2,3 THE ROLE OF THE PROJECT MANAGER,3.1 OVERVIEW\nThe project manager plays a crit...
3,4 PROJECT INTEGRATION MANAGEMENT,Project Integration Management includes the pr...
4,5 PROJECT SCOPE MANAGEMENT,Project Scope Management includes the processe...
5,6 PROJECT SCHEDULE MANAGEMENT,Project Schedule Management includes the proce...
6,7 PROJECT COST MANAGEMENT,Project Cost Management includes the processes...
7,8 PROJECT QUALITY MANAGEMENT,Project Quality Management includes the proces...
8,9 PROJECT RESOURCE MANAGEMENT,Project Resource Management includes the proce...
9,10 PROJECT COMMUNICATIONS MANAGEMENT,Project Communications Management includes the...


In [ ]:
df_chap1=df_chapters.head(1)
df_chap1

,Chapter,Content
0,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE\nProjec...


In [ ]:
content_chap1 = df_chap1["Content"].str.split('\n')
content_chap1

0    [1.1 OVERVIEW AND PURPOSE OF THIS GUIDE, Proje...
Name: Content, dtype: object

In [ ]:
#Chapitre 1  TESTING THE CODE
"""
CHAPTER 1 :
    TESTING THE EXTRACTION OF PDF

"""
import  re
content_chap1 = df_chap1["Content"].str.split('\n')


titles_level1=[] #a.x
titles_level2=[] #a.x.x
titles_level3=[] #a.x.x.x
Contents=[]
i=0
content=""
t1=None
t2=None
t3=None

for line_list in content_chap1:
    for i,line in enumerate(line_list):
        match = re.search(r'^1\.\d+(\.\d+)*\s+', line)
        if line and match :
            previous_line = line_list[i -1]
            level=len(match.group(0).split('.'))-1
            #print(level)
            if level==1:
                #print("LEVEL 1")
                titles_level1.append(line)
                titles_level2.append(None)
                titles_level3.append(None)
                t1=line
                #print(f" title'{t1}'")
                t2=None
                t3=None
            if level==2:
                #print("LEVEL 2")
                titles_level1.append(t1)
                titles_level2.append(line)
                titles_level3.append(None)
                t2=line
                t3=None
            if level==3:
            #print("LEVEL 3")
                titles_level1.append(t1)
                titles_level2.append(t2)
                titles_level3.append(line)
                t3=line
            match2 = re.search(r'^1\.\d+(\.\d+)*\s+', previous_line)
            if match2:
                Contents.append(None)
            # If there is content, append it
            if content:
                Contents.append(content.strip())
                content = ""
        else:
        # Append content lines
            content += '\n' + line
# Append any remaining content
    if content:
        Contents.append(content.strip())
    # Make sure all lists have the same length by filling any missing data with placeholders
    max_length = max(len(titles_level1), len(titles_level2), len(titles_level3), len(Contents))
    titles_level1.extend([""] * (max_length - len(titles_level1)))
    titles_level2.extend([""] * (max_length - len(titles_level2)))
    titles_level3.extend([""] * (max_length - len(titles_level3)))
    Contents.extend([""] * (max_length - len(Contents)))

    df_output = pd.DataFrame({
    'TitleL1': titles_level1,
    'TitleL2': titles_level2,
    'TitleL3': titles_level3,
    'Content': Contents,
    })
df_output


,TitleL1,TitleL2,TitleL3,Content
0,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,None,None,Project management is not new. It has been in ...
1,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.1 THE STANDARD FOR PROJECT MANAGEMENT,None,This guide is based on The Standard for Projec...
2,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.2 COMMON VOCABULARY,None,A common vocabulary is an essential element of...
3,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.3 CODE OF ETHICS AND PROFESSIONAL CONDUCT,None,PMI publishes the Code of Ethics and Professio...
4,1.2 FOUNDATIONAL ELEMENTS,None,None,This section describes foundational elements n...
5,1.2 FOUNDATIONAL ELEMENTS,1.2.1 PROJECTS,None,A project is a temporary endeavor undertaken t...
6,1.2 FOUNDATIONAL ELEMENTS,1.2.2 THE IMPORTANCE OF PROJECT MANAGEMENT,None,Project management is the application of knowl...
7,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",None,None
8,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",1.2.3.1 OVERVIEW,"Using project management processes, tools, and..."
9,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",1.2.3.2 PROGRAM MANAGEMENT,Program management is defined as the applicati...


In [ ]:
# Add new Column for CHAPTER 1 and fix it ;
df_chap1_new=df_output

value_to_fill = df_chapters.iloc[0, 0]
df_chap1_new['Chapter'] = value_to_fill

new_column_name = 'Chapter'
df_chap1_new = df_chap1_new[[new_column_name] + [col for col in df_chap1_new if col != new_column_name]]
df_chap1_new

,Chapter,TitleL1,TitleL2,TitleL3,Content
0,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,None,None,Project management is not new. It has been in ...
1,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.1 THE STANDARD FOR PROJECT MANAGEMENT,None,This guide is based on The Standard for Projec...
2,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.2 COMMON VOCABULARY,None,A common vocabulary is an essential element of...
3,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.3 CODE OF ETHICS AND PROFESSIONAL CONDUCT,None,PMI publishes the Code of Ethics and Professio...
4,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,None,None,This section describes foundational elements n...
5,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,1.2.1 PROJECTS,None,A project is a temporary endeavor undertaken t...
6,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,1.2.2 THE IMPORTANCE OF PROJECT MANAGEMENT,None,Project management is the application of knowl...
7,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",None,None
8,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",1.2.3.1 OVERVIEW,"Using project management processes, tools, and..."
9,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,"1.2.3 RELATIONSHIP OF PROJECT, PROGRAM, PORTFO...",1.2.3.2 PROGRAM MANAGEMENT,Program management is defined as the applicati...


In [ ]:
#ALL THE REST OF THE CHAPTERS FROM 1 TO 13

Final_df = pd.DataFrame({'Chapter': [], 'TitleL1': [], 'TitleL2': [], 'TitleL3': [], 'Content': []})
#Final_df = pd.concat([Final_df,df_chap1_new ], ignore_index=True)

for j in range(1,14):
    df_chap=pd.DataFrame()
    df_chap=df_chapters.iloc[j-1:j]
    content_chap=pd.DataFrame()
    content_chap = df_chap["Content"].str.split('\n')

    titles_level1=[] #a.x
    titles_level2=[] #a.x.x
    titles_level3=[] #a.x.x.x
    t_list=[] #list that contains a.x.x.x parts of all Levels
    Contents=[]
    i=0
    content=""
    t1=None
    t2=None
    t3=None

    for line_list in content_chap:
        for i,line in enumerate(line_list):
            pattern = r'^' + str(j) + r'\.\d+(\.\d+)*\s+'
            match = re.search(pattern, line)
            if line and match and not match.group(0)[:-1] in t_list and "—" not in line:
                #print(t_list)
                previous_line = line_list[i -1]
                level=len(match.group(0).split('.'))-1
                #print(level)

                if len(t_list)==0 and len(Contents)==0 and content:
                    #print(t_list)
                    titles_level1.append(None)
                    titles_level2.append(None)
                    titles_level3.append(None)
                    t1=None
                    t2=None
                    t3=None
                if level==1:
                    #print("LEVEL 1")
                    titles_level1.append(line)
                    titles_level2.append(None)
                    titles_level3.append(None)
                    t1=line
                    t2=None
                    t3=None
                    t1_parts=t1.split(" ")
                    t_list.append(t1_parts[0])
                if level==2:
                    titles_level1.append(t1)
                    titles_level2.append(line)
                    titles_level3.append(None)
                    t2=line
                    t3=None
                    t2_parts=t2.split(" ")
                    t_list.append(t2_parts[0])
                if level==3:
                    titles_level1.append(t1)
                    titles_level2.append(t2)
                    titles_level3.append(line)
                    t3=line
                    t3_parts=t3.split(" ")
                    t_list.append(t3_parts[0])
                match2 = re.search(pattern, previous_line)
                if match2:
                    Contents.append(None)
                # If there is content, append it
                if content :
                    Contents.append(content.strip())
                    content = ""
            else:
            # Append content lines
                content += '\n' + line
    # Append any remaining content
        if content:
            Contents.append(content.strip())
        # Make sure all lists have the same length by filling any missing data with placeholders
        max_length = max(len(titles_level1), len(titles_level2), len(titles_level3), len(Contents))
        titles_level1.extend([""] * (max_length - len(titles_level1)))
        titles_level2.extend([""] * (max_length - len(titles_level2)))
        titles_level3.extend([""] * (max_length - len(titles_level3)))
        Contents.extend([""] * (max_length - len(Contents)))

        df_output = pd.DataFrame({
        'TitleL1': titles_level1,
        'TitleL2': titles_level2,
        'TitleL3': titles_level3,
        'Content': Contents,
        })

    df_chap_new=pd.DataFrame()
    df_chap_new=df_output

    value_to_fill = df_chapters.iloc[j-1, 0]
    df_chap_new['Chapter'] = value_to_fill

    new_column_name = 'Chapter'
    df_chap_new = df_chap_new[[new_column_name] + [col for col in df_chap_new if col != new_column_name]]
    Final_df = pd.concat([Final_df,df_chap_new ], ignore_index=True)
Final_df


,Chapter,TitleL1,TitleL2,TitleL3,Content
0,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,None,None,Project management is not new. It has been in ...
1,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.1 THE STANDARD FOR PROJECT MANAGEMENT,None,This guide is based on The Standard for Projec...
2,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.2 COMMON VOCABULARY,None,A common vocabulary is an essential element of...
3,1 INTRODUCTION,1.1 OVERVIEW AND PURPOSE OF THIS GUIDE,1.1.3 CODE OF ETHICS AND PROFESSIONAL CONDUCT,None,PMI publishes the Code of Ethics and Professio...
4,1 INTRODUCTION,1.2 FOUNDATIONAL ELEMENTS,None,None,This section describes foundational elements n...
...,...,...,...,...,...
935,13 PROJECT STAKEHOLDER MANAGEMENT,13.4 MONITOR STAKEHOLDER ENGAGEMENT,13.4.3 MONITOR STAKEHOLDER ENGAGEMENT: OUTPUTS,None,None
936,13 PROJECT STAKEHOLDER MANAGEMENT,13.4 MONITOR STAKEHOLDER ENGAGEMENT,13.4.3 MONITOR STAKEHOLDER ENGAGEMENT: OUTPUTS,13.4.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...
937,13 PROJECT STAKEHOLDER MANAGEMENT,13.4 MONITOR STAKEHOLDER ENGAGEMENT,13.4.3 MONITOR STAKEHOLDER ENGAGEMENT: OUTPUTS,13.4.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. A change request...
938,13 PROJECT STAKEHOLDER MANAGEMENT,13.4 MONITOR STAKEHOLDER ENGAGEMENT,13.4.3 MONITOR STAKEHOLDER ENGAGEMENT: OUTPUTS,13.4.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...


In [ ]:
Final_df.to_csv("DF_CHAPTERs_SUBTITLEsLv3_CONTENTs_NOT_CLEAN.csv", index=False)

In [ ]:
filtered_df = Final_df[Final_df['Chapter'] == '11 PROJECT RISK MANAGEMENT']

In [ ]:
filtered_df

,Chapter,TitleL1,TitleL2,TitleL3,Content
685,11 PROJECT RISK MANAGEMENT,None,None,None,Project Risk Management includes the processes...
686,11 PROJECT RISK MANAGEMENT,11.1 PLAN RISK MANAGEMENT,None,None,Plan Risk Management is the process of definin...
687,11 PROJECT RISK MANAGEMENT,11.1 PLAN RISK MANAGEMENT,11.1.1 PLAN RISK MANAGEMENT: INPUTS,None,None
688,11 PROJECT RISK MANAGEMENT,11.1 PLAN RISK MANAGEMENT,11.1.1 PLAN RISK MANAGEMENT: INPUTS,11.1.1.1 PROJECT CHARTER,Described in Section 4.1.3.1. The project char...
689,11 PROJECT RISK MANAGEMENT,11.1 PLAN RISK MANAGEMENT,11.1.1 PLAN RISK MANAGEMENT: INPUTS,11.1.1.2 PROJECT MANAGEMENT PLAN,Described in Section 4.2.3.1. In planning Proj...
...,...,...,...,...,...
791,11 PROJECT RISK MANAGEMENT,11.7 MONITOR RISKS,11.7.3 MONITOR RISKS: OUTPUTS,11.7.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...
792,11 PROJECT RISK MANAGEMENT,11.7 MONITOR RISKS,11.7.3 MONITOR RISKS: OUTPUTS,11.7.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. The Monitor Risk...
793,11 PROJECT RISK MANAGEMENT,11.7 MONITOR RISKS,11.7.3 MONITOR RISKS: OUTPUTS,11.7.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...
794,11 PROJECT RISK MANAGEMENT,11.7 MONITOR RISKS,11.7.3 MONITOR RISKS: OUTPUTS,11.7.3.4 PROJECT DOCUMENTS UPDATES,Project documents that may be updated as a res...


In [ ]:
import pandas as pd
df2 = pd.DataFrame(columns=['Concept', 'Description'])



In [ ]:
for i in range(685,685+len(filtered_df)):
  if(filtered_df['TitleL3'][i]!=None):
    concept=filtered_df['TitleL3'][i]
  else:
    if(filtered_df['TitleL2'][i]!=None):
      concept=filtered_df['TitleL2'][i]
    else:
      if(filtered_df['TitleL1'][i]!=None):
        concept=filtered_df['TitleL1'][i]
      else:
          concept=filtered_df['Chapter'][i]
  print(concept)
  new_row = {'Concept': concept, 'Description': filtered_df['Content'][i]}
  df2 = df2.append(new_row, ignore_index=True)



11 PROJECT RISK MANAGEMENT
11.1 PLAN RISK MANAGEMENT
11.1.1 PLAN RISK MANAGEMENT: INPUTS
11.1.1.1 PROJECT CHARTER
11.1.1.2 PROJECT MANAGEMENT PLAN
11.1.1.3 PROJECT DOCUMENTS
11.1.1.4 ENTERPRISE ENVIRONMENTAL FACTORS
11.1.1.5 ORGANIZATIONAL PROCESS ASSETS
11.1.2 PLAN RISK MANAGEMENT: TOOLS AND TECHNIQUES
11.1.2.1 EXPERT JUDGMENT
11.1.2.2 DATA ANALYSIS
11.1.2.3 MEETINGS
11.1.3 PLAN RISK MANAGEMENT: OUTPUTS
11.1.3.1 RISK MANAGEMENT PLAN
11.2 IDENTIFY RISKS
11.2.1 IDENTIFY RISKS: INPUTS
11.2.1.1 PROJECT MANAGEMENT PLAN
11.2.1.2 PROJECT DOCUMENTS
11.2.1.3 AGREEMENTS
11.2.1.4 PROCUREMENT DOCUMENTATION
11.2.1.5 ENTERPRISE ENVIRONMENTAL FACTORS
11.2.1.6 ORGANIZATIONAL PROCESS ASSETS
11.2.2 IDENTIFY RISKS: TOOLS AND TECHNIQUES
11.2.2.1 EXPERT JUDGMENT
11.2.2.2 DATA GATHERING
11.2.2.3 DATA ANALYSIS
11.2.2.4 INTERPERSONAL AND TEAM SKILLS
11.2.2.5 PROMPT LISTS
11.2.2.6 MEETINGS
11.2.3 IDENTIFY RISKS: OUTPUTS
11.2.3.1 RISK REGISTER
11.2.3.2 RISK REPORT
11.2.3.3 PROJECT DOCUMENTS UPDATES
11.3 PERFOR

In [ ]:
filtered_df['TitleL2'][685]

In [ ]:
df2

,Concept,Description
0,11 PROJECT RISK MANAGEMENT,Project Risk Management includes the processes...
1,11.1 PLAN RISK MANAGEMENT,Plan Risk Management is the process of definin...
2,11.1.1 PLAN RISK MANAGEMENT: INPUTS,None
3,11.1.1.1 PROJECT CHARTER,Described in Section 4.1.3.1. The project char...
4,11.1.1.2 PROJECT MANAGEMENT PLAN,Described in Section 4.2.3.1. In planning Proj...
...,...,...
106,11.7.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...
107,11.7.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. The Monitor Risk...
108,11.7.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...
109,11.7.3.4 PROJECT DOCUMENTS UPDATES,Project documents that may be updated as a res...


In [ ]:
df2.to_csv("pmbook_shih.csv", index=False)

In [ ]:
for i in range(685,685+len(filtered_df)):
  print(filtered_df["Content"][i])

Project Risk Management includes the processes of conducting risk management planning, identification, analysis,
response planning, response implementation, and monitoring risk on a project. The objectives of project risk management
are to increase the probability and/or impact of positive risks and to decrease the probability and/or impact of negative
risks, in order to optimize the chances of project success.
The Project Risk Management processes are:
11.1 Plan Risk Management—The process of defining how to conduct risk management activities for a project.
11.2 Identify Risks—The process of identifying individual project risks as well as sources of overall project risk,
and documenting their characteristics.
11.3 Perform Qualitative Risk Analysis—The process of prioritizing individual project risks for further analysis
or action by assessing their probability of occurrence and impact as well as other characteristics.
11.4 Perform Quantitative Risk Analysis—The process of numerically 

In [ ]:
import nltk
nltk.download('all')


import pandas as pd
from keybert import KeyBERT
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Télécharger les ressources NLTK (si ce n'est pas déjà fait)
nltk.download('punkt')

# Créer un objet KeyBERT avec un modèle BERT pré-entraîné
model = KeyBERT('distilbert-base-nli-mean-tokens')

# Initialiser le lemmatiseur WordNet
lemmatizer = WordNetLemmatizer()

# Fonction pour extraire des mots-clés à partir de la description
def extract_keywords(description_text):
    if not isinstance(description_text, str) or not description_text.strip():
        return []  # Retourne une liste vide pour les entrées non-chaînes ou vides

    print(f"Texte d'origine : {description_text}")

    # Extraire les mots-clés à partir de la description en utilisant KeyBERT
    keywords = model.extract_keywords(description_text, keyphrase_ngram_range=(2, 3), stop_words='english')
    extracted_keywords = [keyword[0] for keyword in keywords]

    # Lemmatisation et fusion des mots-clés
    lemmatized_keywords = []
    unique_keywords = set()
    for keyword in extracted_keywords:
        words = word_tokenize(keyword)  # Tokenize the keyword
        lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
        merged_keyword = " ".join(lemmatized_words)
        # Check for duplicates and keep only unique keywords
        if "updating risk" in merged_keyword:
            merged_keyword = merged_keyword.replace("updating risk", "risk updating")
        if merged_keyword not in unique_keywords:
            lemmatized_keywords.append(merged_keyword)
            unique_keywords.add(merged_keyword)
    print(f"Mots-clés lemmatisés et fusionnés : {lemmatized_keywords}")

    return lemmatized_keywords

# Appliquer la fonction extract_keywords à la colonne 'Description' de votre DataFrame
df2['Keywords'] = df2['Description'].apply(extract_keywords)

# Afficher le DataFrame mis à jour avec les mots-clés extraits, lemmatisés et fusionnés
df2


[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package basque_grammars to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping grammars/basque_grammars.zip.
[nltk_data]    | Downloading package bcp47 to /root/nltk_data...
[nltk_data]    | Downloading package biocreative_ppi to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   U

Texte d'origine : Project Risk Management includes the processes of conducting risk management planning, identification, analysis,
response planning, response implementation, and monitoring risk on a project. The objectives of project risk management
are to increase the probability and/or impact of positive risks and to decrease the probability and/or impact of negative
risks, in order to optimize the chances of project success.
The Project Risk Management processes are:
11.1 Plan Risk Management—The process of defining how to conduct risk management activities for a project.
11.2 Identify Risks—The process of identifying individual project risks as well as sources of overall project risk,
and documenting their characteristics.
11.3 Perform Qualitative Risk Analysis—The process of prioritizing individual project risks for further analysis
or action by assessing their probability of occurrence and impact as well as other characteristics.
11.4 Perform Quantitative Risk Analysis—The proce

,Concept,Description,Keywords
0,11 PROJECT RISK MANAGEMENT,Project Risk Management includes the processes...,"[tailoring project risk, shaping project strat..."
1,11.1 PLAN RISK MANAGEMENT,Plan Risk Management is the process of definin...,"[effectiveness determines project, project nec..."
2,11.1.1 PLAN RISK MANAGEMENT: INPUTS,None,[]
3,11.1.1.1 PROJECT CHARTER,Described in Section 4.1.3.1. The project char...,"[project charter document, section project cha..."
4,11.1.1.2 PROJECT MANAGEMENT PLAN,Described in Section 4.2.3.1. In planning Proj...,"[component influence plan, plan taken consider..."
...,...,...,...
106,11.7.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...,"[project risk management, work performance inf..."
107,11.7.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. The Monitor Risk...,"[plan change request, change request include, ..."
108,11.7.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...,"[change project management, organization chang..."
109,11.7.3.4 PROJECT DOCUMENTS UPDATES,Project documents that may be updated as a res...,"[project document updated, risk report updated..."


In [ ]:
df2['Description'] = df2['Description'].fillna('')

In [ ]:
df2

,Concept,Description,Keywords
0,11 PROJECT RISK MANAGEMENT,Project Risk Management includes the processes...,"[tailoring project risk, shaping project strat..."
1,11.1 PLAN RISK MANAGEMENT,Plan Risk Management is the process of definin...,"[effectiveness determines project, project nec..."
2,11.1.1 PLAN RISK MANAGEMENT: INPUTS,,[]
3,11.1.1.1 PROJECT CHARTER,Described in Section 4.1.3.1. The project char...,"[project charter document, section project cha..."
4,11.1.1.2 PROJECT MANAGEMENT PLAN,Described in Section 4.2.3.1. In planning Proj...,"[component influence plan, plan taken consider..."
...,...,...,...
106,11.7.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...,"[project risk management, work performance inf..."
107,11.7.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. The Monitor Risk...,"[plan change request, change request include, ..."
108,11.7.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...,"[change project management, organization chang..."
109,11.7.3.4 PROJECT DOCUMENTS UPDATES,Project documents that may be updated as a res...,"[project document updated, risk report updated..."


In [ ]:
def convert_to_lowercase(text):
    words = text.split()
    updated_words = [word.lower() if word.isupper() else word for word in words]
    return ' '.join(updated_words)

# Apply the function to the 'Concepts' column
df2['Concept'] = df2['Concept'].apply(convert_to_lowercase)

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
def extract_relations_from_model_output(text):
    relations = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    text_replaced = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")
    for token in text_replaced.split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        relations.append({
            'head': subject.strip(),
            'type': relation.strip(),
            'tail': object_.strip()
        })
    return relations

In [ ]:
class KB():
    def __init__(self):
        self.relations = []

    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)

    def add_relation(self, r):
        if not self.exists_relation(r):
            self.relations.append(r)

    def save(self):
        dict_list = []
        for r in self.relations:
            dict_list.append(r)
        return(dict_list)
        #df = pd.DataFrame(dict_list, columns=['head', 'type', 'tail'])
        #print(dict_list)

In [ ]:
def from_small_text_to_kb(text, verbose=False):
    kb = KB()

    # Tokenizer text
    model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True,
                            return_tensors='pt')
    if verbose:
        print(f"Num tokens: {len(model_inputs['input_ids'][0])}")

    # Generate
    gen_kwargs = {
        "max_length": 1000,
        "length_penalty": 0,
        "num_beams": 3,
        "num_return_sequences": 3
    }

    generated_tokens = model.generate(
        **model_inputs,
        **gen_kwargs,
    )

    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

    # create kb
    for sentence_pred in decoded_preds:
        relations = extract_relations_from_model_output(sentence_pred)
        for r in relations:
            kb.add_relation(r)

    return kb

In [ ]:
df2['Description'] = df2['Description'].str.encode('utf-8', 'ignore').str.decode('utf-8')


In [ ]:
import pandas as pd
relation_list = []

# Étape 3 : Appliquer le code à la colonne 'Descr'
for item in tqdm(df2['Description']):
    if isinstance(item, str):  # Vérifier si l'élément est une chaîne de caractères valide
        try:
            kb = from_small_text_to_kb(item, verbose=False) # Remplacez avec votre fonction
            relation_list.append(kb.save())
        except Exception as e:
            print(f"Erreur : {e} pour l'élément '{item}'") # Afficher les erreurs éventuelles
    else:
        print(f"Attention : élément non valide '{item}'")

# Étape 4 : Vérifiez la longueur de la liste et imprimez-la
print(f"Longueur de la liste : {len(relation_list)}")

# Étape 5 : Si la longueur de la liste est correcte, ajoutez les résultats à une nouvelle colonne du DataFrame
if len(relation_list) == len(df2):
    df2['Relation'] = relation_list
else:
    print("Erreur : La longueur de la liste ne correspond pas à la longueur du DataFrame.")

# Étape 6 : Affichez le DataFrame modifié
print(df2)


100%|██████████| 111/111 [08:32<00:00,  4.62s/it]

Longueur de la liste : 111
                                            Concept  \
0                        11 project risk management   
1                         11.1 plan risk management   
2               11.1.1 plan risk management: inputs   
3                          11.1.1.1 project charter   
4                  11.1.1.2 project management plan   
..                                              ...   
106           11.7.3.1 work performance information   
107                        11.7.3.2 change requests   
108        11.7.3.3 project management plan updates   
109              11.7.3.4 project documents updates   
110  11.7.3.5 organizational process assets updates   

                                           Description  \
0    Project Risk Management includes the processes...   
1    Plan Risk Management is the process of definin...   
2                                                        
3    Described in Section 4.1.3.1. The project char...   
4    Described in Sect

In [ ]:
dict_list =[]
for item in tqdm(relation_list):
    for dic in item:
        dict_list.append(dic)
triplet = pd.DataFrame(dict_list, columns=['head', 'type', 'tail'])

100%|██████████| 111/111 [00:00<00:00, 158356.38it/s]


In [ ]:
triplet

,head,type,tail
0,risk management,studies,risk
1,risk,studied by,risk management
2,Project Risk Management,subclass of,risk management
3,Plan Risk Management,use,risk management
4,Plan Risk Management,facet of,risk management
...,...,...,...
336,Assumption log,facet of,Monitor Risks
337,Assumption log,subclass of,risk register
338,risk register,use,risk management
339,risk register,part of,risk management plan


In [ ]:
unique_triplets = triplet.drop_duplicates()

In [ ]:
unique_triplets

,head,type,tail
0,risk management,studies,risk
1,risk,studied by,risk management
2,Project Risk Management,subclass of,risk management
3,Plan Risk Management,use,risk management
4,Plan Risk Management,facet of,risk management
...,...,...,...
333,change control,facet of,project management plan
334,change control process,facet of,project management plan
335,change control,part of,project management plan
336,Assumption log,facet of,Monitor Risks


In [ ]:
tiplet_unique = triplet['head'].unique()

In [ ]:
final_triplet_unique = triplet.drop_duplicates(keep='first')

In [ ]:
final_triplet_unique = final_triplet_unique.reset_index(drop=True)

In [ ]:
final_triplet_unique

,head,type,tail
0,risk management,studies,risk
1,risk,studied by,risk management
2,Project Risk Management,subclass of,risk management
3,Plan Risk Management,use,risk management
4,Plan Risk Management,facet of,risk management
...,...,...,...
252,change control,facet of,project management plan
253,change control process,facet of,project management plan
254,change control,part of,project management plan
255,Assumption log,facet of,Monitor Risks


In [ ]:
text = ''
for item in df2['Description']:
    text += item


In [ ]:
text = ''
for item in df2['Description']:
    text += ''.join(item)
example_sent = text
stop_words = set(stopwords.words('english'))
word_tokens = word_tokenize(example_sent)
filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
filtered_sentence = []

for w in word_tokens:
    if w not in stop_words:
        filtered_sentence.append(w)
test_text = ' '.join(filtered_sentence)

In [ ]:
import textacy
triple_list = []
for sentence in test_text.split('.'):
    t1 = nlp(sentence)
    triple = textacy.extract.subject_verb_object_triples(t1)
    if triple:
        triple_to_list = list(triple)
        triple_list.append(pd.DataFrame(triple_to_list))
svo=pd.concat(triple_list, axis=0) # this should concat all dfs on top of one another using axis=0
svo.columns=['subject','verb','object'] # change your columns on teh final df

In [ ]:
svo

,subject,verb,object
0,"[Project, Risk, Management]",[includes],[processes]
0,"[objectives, project, impact, risks]",[decrease],[risks]
1,[order],[optimize],"[project, success]"
0,[3],[Perform],"[Qualitative, Risk, Analysis]"
1,[process],[risks],"[analysis, action, probability, occurrence, im..."
...,...,...,...
0,[This],"[may, include]","[adding, new, risks, ,, updating, outdated, ri..."
0,[project],[risks],"[project, risk]"
0,"[risk, report]","[may, include]",[details]
0,[It],"[may, include]","[conclusions, effectiveness, management, process]"


In [ ]:
svo = svo.reset_index(drop=True)

In [ ]:
final_svo = copy.deepcopy(svo)

In [ ]:
for idx,row in final_svo.iterrows():
    for c in final_svo.columns:
        for i in row[c]:
            if isinstance(i, spacy.tokens.span.Span):
                row[c] = i.text
            else:
                row[c] = i

In [ ]:
for idx,item in enumerate(final_svo['subject']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['subject'] = item
    else:
        item = item
        final_svo.iloc[idx]['subject'] = item
for idx,item in enumerate(final_svo['verb']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['verb'] = item
    else:
        item = item
        final_svo.iloc[idx]['verb'] = item
for idx,item in enumerate(final_svo['object']):
    if isinstance(item, spacy.tokens.token.Token):
        item = item.text
        final_svo.iloc[idx]['object'] = item
    else:
        item = item
        final_svo.iloc[idx]['object'] = item

In [ ]:
concept_list = []
concept_index = []
for _,row in final_svo.iterrows():
    item = row['subject']
    for i in tiplet_unique:
        if (i in item):
            concept_list.append(item)
            concept_index.append(list(final_svo['subject']).index(item))
svo_unique = dict(zip(concept_index, concept_list))

In [ ]:
final_svo_unique = final_svo.drop_duplicates(keep='first')

In [ ]:
final_svo_unique

,subject,verb,object
0,Management,includes,processes
1,risks,decrease,risks
2,order,optimize,success
3,3,Perform,Analysis
4,process,risks,impact
...,...,...,...
744,risk,register,risks
745,This,include,","
747,report,include,details
748,It,include,process


In [ ]:
df_final['Concepts'] = data_frame['scope_concepts']
df2['Type_relation'] = ''
df2['Concept_of_type_relation'] = ''

NameError: ignored

In [ ]:
final_triplet_unique = final_triplet_unique.reset_index(drop=True)

In [ ]:
final_triplet_unique

,head,type,tail
0,risk management,studies,risk
1,risk,studied by,risk management
2,Project Risk Management,subclass of,risk management
3,Plan Risk Management,use,risk management
4,Plan Risk Management,facet of,risk management
...,...,...,...
252,change control,facet of,project management plan
253,change control process,facet of,project management plan
254,change control,part of,project management plan
255,Assumption log,facet of,Monitor Risks


In [ ]:
for _,row1 in final_triplet_unique.iterrows():
    for _,row2 in df2.iterrows():
        if row1['head'] in row2['Concept']:
            row2['Type_relation'] = row1['type']
            row2['Concept_of_type_relation'] = row1['tail']

In [ ]:
df2

,Concept,Description,Keywords,Relation
0,11 project risk management,Project Risk Management includes the processes...,"[tailoring project risk, shaping project strat...","[{'head': 'risk management', 'type': 'studies'..."
1,11.1 plan risk management,Plan Risk Management is the process of definin...,"[effectiveness determines project, project nec...","[{'head': 'Plan Risk Management', 'type': 'use..."
2,11.1.1 plan risk management: inputs,,[],"[{'head': 'World Heritage Site', 'type': 'main..."
3,11.1.1.1 project charter,Described in Section 4.1.3.1. The project char...,"[project charter document, section project cha...","[{'head': 'requirements', 'type': 'part of', '..."
4,11.1.1.2 project management plan,Described in Section 4.2.3.1. In planning Proj...,"[component influence plan, plan taken consider...","[{'head': 'Project Risk Management', 'type': '..."
...,...,...,...,...
106,11.7.3.1 work performance information,Described in Section 4.5.1.3. Work performance...,"[project risk management, work performance inf...","[{'head': 'Work performance information', 'typ..."
107,11.7.3.2 change requests,Described in Section 4.3.3.4. The Monitor Risk...,"[plan change request, change request include, ...","[{'head': 'Monitor Risks', 'type': 'facet of',..."
108,11.7.3.3 project management plan updates,Any change to the project management plan goes...,"[change project management, organization chang...","[{'head': 'change control', 'type': 'facet of'..."
109,11.7.3.4 project documents updates,Project documents that may be updated as a res...,"[project document updated, risk report updated...","[{'head': 'Assumption log', 'type': 'facet of'..."


In [ ]:
df2['reference'] = None
df2['links']= None

In [ ]:
import pandas as pd
# Replace 'figre' with 'figure' in the 'Text' column
df2['Description'] = df2['Description'].str.replace('figre', 'figure')

# Print the updated DataFrame
print(df2)

                                            Concept  \
0                        11 project risk management   
1                         11.1 plan risk management   
2               11.1.1 plan risk management: inputs   
3                          11.1.1.1 project charter   
4                  11.1.1.2 project management plan   
..                                              ...   
106           11.7.3.1 work performance information   
107                        11.7.3.2 change requests   
108        11.7.3.3 project management plan updates   
109              11.7.3.4 project documents updates   
110  11.7.3.5 organizational process assets updates   

                                           Description  \
0    Project Risk Management includes the processes...   
1    Plan Risk Management is the process of definin...   
2                                                        
3    Described in Section 4.1.3.1. The project char...   
4    Described in Section 4.2.3.1. In planning Pr

In [ ]:
df2['Description'][33]

'Perform Qualitative Risk Analysis is the process of prioritizing individual project risks for further analysis or actionby assessing their probability of occurrence and impact as well as other characteristics. The key benefit of thisprocess is that it focuses efforts on high-priority risks. This process is performed throughout the project. The inputs,tools and techniques, and outputs of the process are depicted in Figure 11-8. Figure 11-9 depicts the data flowdiagram for the process.NubPage 385Perform Qualitative Risk Analysis assesses the priority of identified individual project risks using their probability ofoccurrence, the corresponding impact on project objectives if the risks occur, and other factors. Such assessments aresubjective as they are based on perceptions of risk by the project team and other stakeholders. Effective assessmenttherefore requires explicit identification and management of the risk attitudes of key participants in the PerformQualitative Risk Analysis proce

In [ ]:
df2['Description'] = df2['Description'].str.replace('\n', '')
df2['Description'] = df2['Description'].str.replace('theProject', 'the Project')
df2['Description'] = df2['Description'].str.replace('◆', '-')

In [ ]:
pattern = r'Figure\d+-\d+'

# Nouveau motif à remplacer
new_pattern = r'Figure \\d+-\\d+'

# Remplacer le motif initial par le nouveau motif
df2['Description'] = df2['Description'].str.replace(pattern, new_pattern)
df2

,Concept,Description,Keywords,Relation,reference,links
0,11 project risk management,Project Risk Management includes the processes...,"[tailoring project risk, shaping project strat...","[{'head': 'risk management', 'type': 'studies'...",[11-1],None
1,11.1 plan risk management,Plan Risk Management is the process of definin...,"[effectiveness determines project, project nec...","[{'head': 'Plan Risk Management', 'type': 'use...",[11-3],None
2,11.1.1 plan risk management: inputs,,[],"[{'head': 'World Heritage Site', 'type': 'main...",[],None
3,11.1.1.1 project charter,Described in Section 4.1.3.1. The project char...,"[project charter document, section project cha...","[{'head': 'requirements', 'type': 'part of', '...",[],None
4,11.1.1.2 project management plan,Described in Section 4.2.3.1. In planning Proj...,"[component influence plan, plan taken consider...","[{'head': 'Project Risk Management', 'type': '...",[],None
...,...,...,...,...,...,...
106,11.7.3.1 work performance information,Described in Section 4.5.1.3. Work performance...,"[project risk management, work performance inf...","[{'head': 'Work performance information', 'typ...",[],None
107,11.7.3.2 change requests,Described in Section 4.3.3.4. The Monitor Risk...,"[plan change request, change request include, ...","[{'head': 'Monitor Risks', 'type': 'facet of',...",[],None
108,11.7.3.3 project management plan updates,Any change to the project management plan goes...,"[change project management, organization chang...","[{'head': 'change control', 'type': 'facet of'...",[],None
109,11.7.3.4 project documents updates,Project documents that may be updated as a res...,"[project document updated, risk report updated...","[{'head': 'Assumption log', 'type': 'facet of'...",[],None


In [ ]:
from numpy.core.multiarray import empty
import re

def extraire_phrases_figure(texte):
    # Utiliser une expression régulière pour trouver les phrases commençant par "Figure"
    phrases_figure = re.findall(r'Figure \d+-\d+', texte)
    return phrases_figure

for i in range(len(df2)):
  empty_list = []
  phrases_figure = extraire_phrases_figure(df2['Description'][i])

# Afficher les phrases extraites
  for phrase in phrases_figure:
      phrase=phrase.replace("Figure ","")
      phrase=phrase.replace(" ","")
      empty_list.append(phrase)
      print(phrase)
  if(phrases_figure!=empty):
    df2['reference'][i]=empty_list

11-1
11-3
11-4
11-5
11-6
11-7
11-4
11-8
11-9
11-4
11-5
11-10
11-11
11-12
11-13
11-14
11-15
11-16
11-17
11-19
11-20
11-21


In [ ]:
phrases_figure = extraire_phrases_figure(df2['Description'][25])

In [ ]:
phrases_figure

['Figure 11-2', 'Figure 11-3']

In [ ]:
!pip install rouge
!pip install textacy
!pip install pdfplumber
!pip install rdflib
!pip install bert-score
!pip install autocorrect
!pip install sentence-transformers
!pip install faiss-cpu --no-cache

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Remplacez 'facebook/bart-large-cnn' par le modèle que vous souhaitez utiliser
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def generate_better_description(description):
    if not pd.isnull(description) and description != "":
        input_text = description
        input_ids = tokenizer.encode(input_text, return_tensors="pt", max_length=1024, truncation=True, padding=True)
        output = model.generate(input_ids, max_length=100, num_return_sequences=1, no_repeat_ngram_size=2, top_k=50, top_p=0.95)
        # Extrait la description améliorée du résultat de la génération
        description_amelioree = tokenizer.decode(output[0], skip_special_tokens=True)
        return description_amelioree
    else:
        # Gérez le cas où la description est manquante
        return description

df2['DescriptionAmelioree'] = df2['Description'].apply(generate_better_description)
df2

,Concept,Description,Keywords,DescriptionAmelioree
0,11 PROJECT RISK MANAGEMENT,Project Risk Management includes the processes...,"[tailoring project risk, shaping project strat...",Project Risk Management includes the processes...
1,11.1 PLAN RISK MANAGEMENT,Plan Risk Management is the process of definin...,"[effectiveness determines project, project nec...",Plan Risk Management is the process of definin...
2,11.1.1 PLAN RISK MANAGEMENT: INPUTS,None,[],None
3,11.1.1.1 PROJECT CHARTER,Described in Section 4.1.3.1. The project char...,"[project charter document, section project cha...",Described in Section 4.1.3. The project charte...
4,11.1.1.2 PROJECT MANAGEMENT PLAN,Described in Section 4.2.3.1. In planning Proj...,"[component influence plan, plan taken consider...",Described in Section 4.2.3. In planning Projec...
...,...,...,...,...
106,11.7.3.1 WORK PERFORMANCE INFORMATION,Described in Section 4.5.1.3. Work performance...,"[project risk management, work performance inf...",Work performance information includes informat...
107,11.7.3.2 CHANGE REQUESTS,Described in Section 4.3.3.4. The Monitor Risk...,"[plan change request, change request include, ...",The Monitor Risks process may result in a chan...
108,11.7.3.3 PROJECT MANAGEMENT PLAN UPDATES,Any change to the project management plan goes...,"[change project management, organization chang...",Any change to the project management plan goes...
109,11.7.3.4 PROJECT DOCUMENTS UPDATES,Project documents that may be updated as a res...,"[project document updated, risk report updated...",Project documents that may be updated as a res...
